In [2]:
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

warnings.filterwarnings("ignore", category=UserWarning)

SR = 16_000
DURATION = 2.0
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 256
BATCH_SIZE = 4
SEED = 42
DATA_DIR = Path("data")
FLOAT_MODEL_PATH = Path("tiny_cnn_model_big.tflite")
INT8_MODEL_PATH = Path("licht_klassifikator_int8_big.tflite")
NUM_CLASSES = 3
LABELS = {0: "licht_an", 1: "licht_aus", 2: "unknown"}
CLASS_DIRS = {
    0: "licht_an",
    1: "licht_aus",
    2: "unknown",
}
LICHT_AUS_MARGIN = 0.08
MIN_COMMAND_CONFIDENCE = 0.50
LICHT_AN_OVERSAMPLE = 2
TRIM_TOP_DB = 25
MIN_SIGNAL_RMS = 0.01

np.random.seed(SEED)
tf.random.set_seed(SEED)



In [2]:
# ========== 1. DATEN LADEN ==========

def collect_wav_files(class_dir: Path) -> list[Path]:
    files = sorted(class_dir.glob("*.wav"))
    if not files:
        files = sorted(class_dir.rglob("*.wav"))
    return files


class_files = {
    label_idx: collect_wav_files(DATA_DIR / folder_name)
    for label_idx, folder_name in CLASS_DIRS.items()
}

missing = [CLASS_DIRS[idx] for idx, files in class_files.items() if not files]
if missing:
    raise FileNotFoundError(
        "Keine WAV-Dateien gefunden. Erwartet: "
        + ", ".join(f"data/{name}/*.wav" for name in CLASS_DIRS.values())
        + f". Fehlend: {', '.join(missing)}"
    )

print(" | ".join(f"{CLASS_DIRS[idx]}: {len(files)}" for idx, files in class_files.items()))


def normalize_audio(audio: np.ndarray) -> np.ndarray:
    audio = np.nan_to_num(np.asarray(audio, dtype=np.float32).flatten())
    peak = float(np.max(np.abs(audio)))
    if peak > 1e-6:
        audio = audio / peak
    return audio


def prepare_audio_clip(
    audio: np.ndarray,
    sr: int = SR,
    duration: float = DURATION,
    trim_silence: bool = False,
    top_db: int = TRIM_TOP_DB,
) -> np.ndarray:
    """Gemeinsame Vorverarbeitung fuer Training und Live-Mikrofon."""
    audio = normalize_audio(audio)
    if trim_silence:
        trimmed, _ = librosa.effects.trim(audio, top_db=top_db)
        if len(trimmed) > 0:
            audio = trimmed

    target_len = int(sr * duration)
    if len(audio) > target_len:
        start = (len(audio) - target_len) // 2
        audio = audio[start : start + target_len]
    elif len(audio) < target_len:
        pad_total = target_len - len(audio)
        audio = np.pad(audio, (pad_total // 2, pad_total - pad_total // 2))

    return audio.astype(np.float32)


def load_audio(path, sr=SR, duration=DURATION):
    """Lädt Audio und bringt es auf die Modelllaenge."""
    audio, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    return prepare_audio_clip(audio, sr=sr, duration=duration, trim_silence=False)


licht_an: 165 | licht_aus: 165 | unknown: 81


In [3]:
# ========== 2. SPEKTROGRAMME ERSTELLEN ==========

def get_mel_spec(audio, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Erstellt ein normiertes Log-Mel-Spektrogramm."""
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length,
        center=False,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    return log_mel[..., np.newaxis].astype(np.float32)


def pad_or_crop_time(spec, target_width):
    """Bringt alle Spektrogramme auf dieselbe Zeitachsen-Länge."""
    width = spec.shape[1]
    if width > target_width:
        start = (width - target_width) // 2
        spec = spec[:, start : start + target_width, :]
    elif width < target_width:
        spec = np.pad(spec, ((0, 0), (0, target_width - width), (0, 0)), mode="constant")
    return spec


# Zielbreite aus erstem Sample ableiten
first_file = class_files[0][0]
reference_width = get_mel_spec(load_audio(first_file)).shape[1]

X, y = [], []
for label_idx, files in class_files.items():
    repeat = LICHT_AN_OVERSAMPLE if label_idx == 0 else 1
    for f in files:
        spec = pad_or_crop_time(get_mel_spec(load_audio(f)), reference_width)
        for _ in range(repeat):
            X.append(spec)
            y.append(label_idx)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
INPUT_SHAPE = tuple(X.shape[1:])

print(f"X shape: {X.shape}")
print(f"INPUT_SHAPE: {INPUT_SHAPE}")


c:\Users\David\Studium\AKIB4\VerteilteSysteme\Sprachsteuerung\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X shape: (576, 64, 124, 1)
INPUT_SHAPE: (64, 124, 1)


In [4]:
# ========== 3. ULTRA-KOMPAKTES CNN ==========
# Ziel: < 3,5 MB = ca. 875.000 Parameter (float32 = 4 Bytes)

def build_tiny_cnn(input_shape=INPUT_SHAPE):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(48, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(72, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(96, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(96, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ])


def compute_class_weights(y_array):
    counts = np.bincount(y_array, minlength=NUM_CLASSES)
    total = len(y_array)
    weights = {
        idx: total / (NUM_CLASSES * count)
        for idx, count in enumerate(counts)
        if count > 0
    }
    weights[0] *= 1.3
    return weights


def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def predict_from_probs(
    probs,
    licht_aus_margin=LICHT_AUS_MARGIN,
    min_confidence=MIN_COMMAND_CONFIDENCE,
):
    probs = np.asarray(probs, dtype=np.float32).reshape(-1)
    p_an, p_aus, p_unk = probs[0], probs[1], probs[2]

    if p_unk >= max(p_an, p_aus) and p_unk >= min_confidence:
        pred_idx = 2
    elif p_aus > p_an + licht_aus_margin:
        pred_idx = 1
    else:
        pred_idx = 0

    if pred_idx != 2 and probs[pred_idx] < min_confidence and p_unk >= probs[pred_idx]:
        pred_idx = 2

    return {
        "predicted_label": LABELS[pred_idx],
        "predicted_idx": pred_idx,
        "confidence": float(probs[pred_idx]),
        "probabilities": {LABELS[i]: float(probs[i]) for i in range(NUM_CLASSES)},
    }


model = compile_model(build_tiny_cnn())
model.summary()

total_params = model.count_params()
model_size_mb = (total_params * 4) / (1024 * 1024)
print(f"\nModell-Parameter: {total_params:,}")
print(f"Geschätzte Größe (float32): {model_size_mb:.2f} MB")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 124, 48)    │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 124, 48)    │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 62, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 62, 72)     │        31,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 62, 72)     │           288 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 31, 72)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 31, 96)     │        62,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 31, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 15, 96)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 96)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           291 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 104,427 (407.92 KB)

 Trainable params: 103,995 (406.23 KB)

 Non-trainable params: 432 (1.69 KB)


Modell-Parameter: 104,427
Geschätzte Größe (float32): 0.40 MB


In [ ]:
# ========== 4. TRAINING MIT DATA AUGMENTATION ==========
# K-Fold Cross-Validation + Augmentation auf Spektrogramm-Ebene

datagen = ImageDataGenerator(
    width_shift_range=0.12,
    height_shift_range=0.12,
    zoom_range=0.12,
    horizontal_flip=False,
)

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold + 1}/10 ---")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    fold_class_weights = compute_class_weights(y_train)

    fold_model = compile_model(build_tiny_cnn())
    history = fold_model.fit(
        datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED + fold),
        steps_per_epoch=max(1, len(X_train) // BATCH_SIZE),
        validation_data=(X_val, y_val),
        class_weight=fold_class_weights,
        epochs=50,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=10, restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=5
            ),
        ],
        verbose=0,
    )

    val_acc = max(history.history["val_accuracy"])
    scores.append(val_acc)
    print(f"Fold {fold + 1} Best Val-Accuracy: {val_acc:.2%}")

print("\n=== Ergebnis ===")
print(f"Durchschnittliche Accuracy: {np.mean(scores):.2%} (+/- {np.std(scores):.2%})")



--- Fold 1/10 ---
Fold 1 Best Val-Accuracy: 67.24%

--- Fold 2/10 ---


In [ ]:
# ========== 5. FINALES MODELL TRAINIEREN ==========

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

class_weights = compute_class_weights(y_train)
print("Class weights:", {LABELS[k]: round(v, 3) for k, v in class_weights.items()})

final_model = compile_model(build_tiny_cnn())
final_history = final_model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    steps_per_epoch=max(1, len(X_train) // BATCH_SIZE),
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    epochs=60,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=7
        ),
    ],
    verbose=1,
)

val_acc = max(final_history.history["val_accuracy"])
print(f"Beste Val-Accuracy (final): {val_acc:.2%}")


Class weights: {'licht_an': np.float64(0.758), 'licht_aus': np.float64(1.162), 'unknown': np.float64(2.359)}
Epoch 1/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.4457 - loss: 1.2622 - val_accuracy: 0.5776 - val_loss: 0.9872 - learning_rate: 0.0010
Epoch 2/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.4609 - loss: 1.1648 - val_accuracy: 0.5776 - val_loss: 0.9879 - learning_rate: 0.0010
Epoch 3/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.4913 - loss: 1.1809 - val_accuracy: 0.5776 - val_loss: 0.9835 - learning_rate: 0.0010
Epoch 4/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.5109 - loss: 1.1110 - val_accuracy: 0.4655 - val_loss: 1.0403 - learning_rate: 0.0010
Epoch 5/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.4696 - loss: 1.1612 - val_accuracy: 0.5517 - val_loss: 0.9766 - learning_rate: 0.0010
Epoch 6/100
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.5500 - loss: 1.1245 - val_accuracy: 0.2845 - val_loss

In [ ]:
# ========== 5b. FEHLERANALYSE LICHT_AN vs LICHT_AUS ==========

from sklearn.metrics import classification_report, confusion_matrix

val_probs = final_model.predict(X_val, verbose=0)
val_preds = np.array([predict_from_probs(p)["predicted_idx"] for p in val_probs])

print("Confusion Matrix (Validation):")
print(confusion_matrix(y_val, val_preds, labels=list(range(NUM_CLASSES))))
print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_preds,
        target_names=[LABELS[i] for i in range(NUM_CLASSES)],
        digits=3,
    )
)

raw_preds = np.argmax(val_probs, axis=1)
an_mask = y_val == 0
aus_as_an = int(np.sum((y_val == 1) & (val_preds == 0)))
an_as_aus = int(np.sum(an_mask & (val_preds == 1)))
raw_an_as_aus = int(np.sum(an_mask & (raw_preds == 1)))

print(f"\nVerwechslungen auf Validation:")
print(f"  licht_an -> licht_aus (ohne Marge): {raw_an_as_aus} / {int(np.sum(an_mask))}")
print(f"  licht_an -> licht_aus (mit Marge {LICHT_AUS_MARGIN}): {an_as_aus} / {int(np.sum(an_mask))}")
print(f"  licht_aus -> licht_an: {aus_as_an} / {int(np.sum(y_val == 1))}")
print("\nHinweis: Wenn live alles falsch ist, liegt es meist an der Mikrofon-Vorverarbeitung, nicht an der Marge.")

Confusion Matrix (Validation):
[[65  1  1]
 [ 0 32  1]
 [ 1  9  6]]

Classification Report:
              precision    recall  f1-score   support

    licht_an      0.985     0.970     0.977        67
   licht_aus      0.762     0.970     0.853        33
     unknown      0.750     0.375     0.500        16

    accuracy                          0.888       116
   macro avg      0.832     0.772     0.777       116
weighted avg      0.889     0.888     0.876       116


Verwechslungen auf Validation:
  licht_an -> licht_aus (ohne Marge): 1 / 67
  licht_an -> licht_aus (mit Marge 0.08): 1 / 67
  licht_aus -> licht_an: 0 / 33

Tipp: Wenn live noch zu oft 'licht_aus' kommt, LICHT_AUS_MARGIN in Zelle 1 erhoehen (z.B. 0.12).


In [ ]:
# ========== 6. FLOAT32 TFLITE EXPORT ==========

def convert_to_tflite(keras_model, optimize=False, representative_data=None):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    if optimize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if representative_data is not None:
            converter.representative_dataset = representative_data
    return converter.convert()


def representative_dataset():
    for i in range(min(20, len(X))):
        yield [X[i : i + 1].astype(np.float32)]


tflite_float = convert_to_tflite(final_model)
FLOAT_MODEL_PATH.write_bytes(tflite_float)

label_mapping = {str(idx): name for idx, name in LABELS.items()}
Path("label_mapping.json").write_text(
    json.dumps(label_mapping, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"Float32-Modell gespeichert: {FLOAT_MODEL_PATH} ({len(tflite_float) / 1024:.1f} KB)")
print(f"Label-Mapping gespeichert: label_mapping.json -> {label_mapping}")


INFO:tensorflow:Assets written to: C:\Users\David\AppData\Local\Temp\tmp72d3b2mn\assets


INFO:tensorflow:Assets written to: C:\Users\David\AppData\Local\Temp\tmp72d3b2mn\assets


Saved artifact at 'C:\Users\David\AppData\Local\Temp\tmp72d3b2mn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 124, 1), dtype=tf.float32, name='keras_tensor_184')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2173702626896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702626512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683330960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683329040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702624976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702628240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683328656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683330768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683329232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683326544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  21

In [ ]:
# ========== 7. QUANTISIERTES TFLITE MODELL ==========
# Dynamic-Range-Quantisierung ist stabiler als Full-INT8 auf Windows.

tflite_quant = convert_to_tflite(
    final_model,
    optimize=True,
    representative_data=representative_dataset,
)
INT8_MODEL_PATH.write_bytes(tflite_quant)

size_kb = len(tflite_quant) / 1024
size_mb = size_kb / 1024
print(f"Quantisiertes Modell gespeichert: {INT8_MODEL_PATH}")
print(f"Größe: {size_kb:.1f} KB ({size_mb:.2f} MB)")
print(f"Gespart gegenüber float32-Parametergröße: {model_size_mb / max(size_mb, 1e-6):.1f}x")


INFO:tensorflow:Assets written to: C:\Users\David\AppData\Local\Temp\tmpus02m2we\assets


INFO:tensorflow:Assets written to: C:\Users\David\AppData\Local\Temp\tmpus02m2we\assets


Saved artifact at 'C:\Users\David\AppData\Local\Temp\tmpus02m2we'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 124, 1), dtype=tf.float32, name='keras_tensor_184')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2173702626896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702626512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683330960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683329040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702624976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173702628240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683328656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683330768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683329232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2173683326544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  21

In [ ]:
# ========== 8. INFERENZ TEST ==========

def prepare_tflite_input(data, input_detail):
    data = np.asarray(data, dtype=np.float32)
    scale, zero_point = input_detail["quantization"]
    dtype = input_detail["dtype"]

    if dtype == np.float32:
        return data.astype(np.float32)
    if scale == 0:
        return data.astype(dtype)

    return np.round(data / scale + zero_point).astype(dtype)


def decode_tflite_output(output, output_detail):
    probs = np.asarray(output, dtype=np.float32).reshape(-1)
    scale, zero_point = output_detail["quantization"]
    if output_detail["dtype"] != np.float32 and scale != 0:
        probs = (probs - zero_point) * scale
    return probs


def run_tflite_inference(model_bytes, sample):
    interpreter = tf.lite.Interpreter(model_content=model_bytes)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    test_input = prepare_tflite_input(sample, input_details[0])
    interpreter.set_tensor(input_details[0]["index"], test_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])
    return decode_tflite_output(output, output_details[0])


for model_name, model_bytes in [
    ("Float32", tflite_float),
    ("Quantisiert", tflite_quant),
]:
    probs = run_tflite_inference(model_bytes, X[0:1])
    result = predict_from_probs(probs)
    true_label = LABELS[int(y[0])]
    print(f"\n{model_name}-Modell:")
    print(f"  Wahr: {true_label}")
    print(f"  Vorhersage: {result['predicted_label']} ({result['confidence']:.1%})")
    print("  Wahrscheinlichkeiten:", ", ".join(
        f"{name}={prob:.3f}" for name, prob in result["probabilities"].items()
    ))

print(f"\nModelle gespeichert unter:")
print(f"  - {FLOAT_MODEL_PATH.resolve()}")
print(f"  - {INT8_MODEL_PATH.resolve()}")



Float32-Modell:
  Wahr: licht_an
  Vorhersage: licht_an (97.4%)
  Wahrscheinlichkeiten: licht_an=0.974, licht_aus=0.004, unknown=0.022

Quantisiert-Modell:
  Wahr: licht_an
  Vorhersage: licht_an (98.8%)
  Wahrscheinlichkeiten: licht_an=0.988, licht_aus=0.004, unknown=0.008

Modelle gespeichert unter:
  - C:\Users\David\Studium\AKIB4\VerteilteSysteme\Sprachsteuerung\tiny_cnn_model_big.tflite
  - C:\Users\David\Studium\AKIB4\VerteilteSysteme\Sprachsteuerung\licht_klassifikator_int8_big.tflite


## Live-Test mit Mikrofon

Das Modell klassifiziert drei Klassen: `licht_an`, `licht_aus` und `unknown`.

Wichtig fuer gute Live-Ergebnisse:
- **Enter druecken und sofort sprechen** (nicht erst warten)
- Live-Audio wird wie im MFCC-Notebook **normalisiert** und **Stille entfernt**
- Der Pipeline-Check oben muss `licht_an` erkennen, bevor der Mikrofon-Test Sinn macht

Vor dem Live-Test die Zellen 1–2, 7–9 und 11 ausfuehren.

In [ ]:
# ========== 9. LIVE-TEST MIT MIKROFON ==========
import sounddevice as sd

LIVE_MODEL_PATH = FLOAT_MODEL_PATH
LIVE_SECONDS = DURATION
REFERENCE_WIDTH = reference_width

if not LIVE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Modell nicht gefunden: {LIVE_MODEL_PATH.resolve()}")

tflite_live_bytes = LIVE_MODEL_PATH.read_bytes()


def audio_rms(audio: np.ndarray) -> float:
    audio = np.asarray(audio, dtype=np.float32).flatten()
    if len(audio) == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(audio))))


def audio_array_to_model_input(audio: np.ndarray) -> np.ndarray:
    prepared = prepare_audio_clip(audio, trim_silence=True)
    spec = pad_or_crop_time(get_mel_spec(prepared), REFERENCE_WIDTH)
    return spec[np.newaxis, ...].astype(np.float32)


def classify_live_audio(audio: np.ndarray) -> dict:
    sample = audio_array_to_model_input(audio)
    probs = run_tflite_inference(tflite_live_bytes, sample)
    result = predict_from_probs(probs)
    result["signal_rms"] = audio_rms(audio)
    return result


def record_microphone(seconds: float = LIVE_SECONDS, sr: int = SR) -> np.ndarray:
    print(f">> JETZT sprechen ({seconds:.1f}s) ...")
    audio = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32")
    sd.wait()
    return audio[:, 0]


# Sanity-Check: Trainingsdatei durch dieselbe Live-Pipeline
sanity_path = class_files[0][0]
sanity_result = classify_live_audio(load_audio(sanity_path))
print("Pipeline-Check (licht_an Trainingsdatei):")
print(
    f"  -> {sanity_result['predicted_label']} "
    f"({sanity_result['confidence']:.1%}) | "
    + ", ".join(f"P({k})={v:.3f}" for k, v in sanity_result["probabilities"].items())
)

print("\nLive-Test bereit.")
print(f"Modell: {LIVE_MODEL_PATH.resolve()}")
print(f"Eingabeform: (1, {N_MELS}, {REFERENCE_WIDTH}, 1) | Sample Rate: {SR} Hz")
print("Enter druecken und den Befehl SOFORT mitsprechen | Strg+C = beenden")

try:
    while True:
        input(">> Enter ...")
        recorded_audio = record_microphone()
        result = classify_live_audio(recorded_audio)
        prob_text = ", ".join(
            f"P({name})={prob:.3f}"
            for name, prob in result["probabilities"].items()
        )
        status = ""
        if result["signal_rms"] < MIN_SIGNAL_RMS:
            status = " | WARNUNG: Signal zu leise"
        print(
            f"   -> {result['predicted_label']} "
            f"({result['confidence']:.1%}) | {prob_text}"
            f"{status}\n"
        )
except KeyboardInterrupt:
    print("Live-Test beendet.")

NameError: name 'prepare_audio_clip' is not defined

In [5]:
import tensorflow as tf

interpreter = tf.lite.Interpreter("tiny_cnn_model_big.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("=== INPUT ===")
print(f"Shape:  {input_details['shape']}")
print(f"Dtype:  {input_details['dtype']}")
print(f"Name:   {input_details['name']}")

print("\n=== OUTPUT ===")
print(f"Shape:  {output_details['shape']}")
print(f"Dtype:  {output_details['dtype']}")

print("\n=== OPS (für TFLite Micro) ===")
ops = set(op['op_name'] for op in interpreter._get_ops_details())
for op in sorted(ops):
    print(f'resolver.Add{op}();')

=== INPUT ===
Shape:  [  1  64 124   1]
Dtype:  <class 'numpy.float32'>
Name:   serving_default_keras_tensor_99:0

=== OUTPUT ===
Shape:  [1 3]
Dtype:  <class 'numpy.float32'>

=== OPS (für TFLite Micro) ===
resolver.AddADD();
resolver.AddCONV_2D();
resolver.AddDELEGATE();
resolver.AddFULLY_CONNECTED();
resolver.AddMAX_POOL_2D();
resolver.AddMEAN();
resolver.AddMUL();
resolver.AddSOFTMAX();
